# DEXTRA · 06 — The severe-day gap: what a reviewer will ask first

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aduitya9/DEXTRA/blob/main/notebooks/06_severe_day_gap.ipynb)

Notebook 05 found what the paper is really about. On the cities a model trained on, it catches severe days (AQI above 300) better than persistence once its alarm level is set on validation. On a city it has never seen it catches 20 of 114 where persistence catches 45: on severe days it sits **72 AQI points below the truth** (32 on the cities it knows), and the further ahead it looks the more it forecasts a return to normal that does not come — 14, then 4, then 2 of 38.

Before building C-STDA to fix that, the three questions a reviewer would ask first:

1. **Do foundation models already solve it?** Chronos-2 and Chronos-Bolt forecast a series they have never seen with no training at all — exactly an unseen city's situation.
2. **Do the simple fixes close it?** Oversampling severe windows, weighting severe days in the loss, training for the 90th percentile, and tuning the alarm level on validation.
3. **Is the gap real, or five cities' bad luck?** The main split has five unseen cities and about forty severe days in their 2025. Leave-one-city-out holds out each of the fifteen cities in turn and scores 2024 *and* 2025, so every city is unseen once, with two winters each.

### What "closing the gap" means — fixed before any result is read
The primary measure is **severe-day F1 on held-out cities, pooled over the fifteen leave-one-city-out folds**, with each forecaster's alarm level chosen on its own validation rows. The bar is **persistence at whichever of its two alarm rules does better** — the official 300 line, as an agency would run it, or a level tuned on validation — so the baseline is never weakened by tuning that does not transfer. A forecaster beats it if **the 95% interval of the difference lies above zero**, resampling whole cities, and the size of the difference is always shown beside that verdict. Everything else here is supporting evidence.

### Runtime
**GPU (T4), about two hours**: three foundation-model setups (about 25 minutes, including a 1.3 GB download — the weather variant is the slow one, since it reads 23 series per forecast), nine training runs on the main split, and 180 across the fifteen folds (about 90 minutes). Every piece is saved the moment it finishes, so a disconnect costs at most the piece in progress.

In [ ]:
# Leave empty to read from the HF_DATASET_REPO Colab secret.
HF_DATASET_REPO = ""
GITHUB_REPO_URL = "https://github.com/Aduitya9/DEXTRA.git"

SEEDS = [42, 43, 44]
# The cheap fixes for rare days, each trained on notebook 05's all-token head (section C).
FIXES = ["oversampled", "weighted", "quantile"]

# Architecture and training: notebook 05's, so the all-token head is one model throughout.
D_MODEL, N_HEADS, N_LAYERS, DROPOUT = 128, 8, 2, 0.3
EPOCHS, BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY, PATIENCE = 30, 256, 3e-4, 1e-3, 10

# Foundation models, zero-shot: (name, Hugging Face id, reads the weather too?)
FOUNDATION = [("chronos-bolt", "amazon/chronos-bolt-base", False),
              ("chronos-2", "amazon/chronos-2", False),
              ("chronos-2-weather", "amazon/chronos-2", True)]
FM_CONTEXT = 512            # days of a city's own history a foundation model reads

LOCO_CITIES = None          # None holds out all fifteen, one at a time

# Safety
TIME_BUDGET_HOURS = 10.5    # stop starting new pieces this far into the Colab session
CHECKPOINT_EVERY_MINUTES = 4

SAVE_TO_HF = True

### If Colab disconnects
The same protection as notebook 05, and the same honest limit: free Colab stops when the tab closes, and this notebook does not fake activity to prevent that. Instead, nothing finished is ever lost:

- **Each piece is saved the moment it finishes**, in one commit with everything that belongs to it: a foundation model's forecasts, a main-split run with its weights and forecasts, a whole leave-one-city-out fold.
- **Re-running skips every finished piece** and carries on with the next. A main-split run cut off part-way resumes from its last checkpoint; a fold cut off part-way starts that fold again — about six minutes on a T4.
- Hugging Face reads and writes are retried on network errors, and the whole notebook makes about thirty commits, well inside the Hub's limits.

So if you are disconnected: reconnect, **Run all**, and it carries on. If it says the session is near Colab's 12-hour limit, use **Runtime → Disconnect and delete runtime**, then **Run all**.

**Keep the tab open and your computer awake** while it runs (Windows: *Settings → System → Power → Screen and sleep → Never*).

## Secrets
`HF_TOKEN` and `HF_DATASET_REPO`, as in the earlier notebooks.

In [ ]:
from google.colab import userdata

def colab_secret(name, required=True):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(
            f"Colab secret '{name}' is missing or this notebook can't read it. Open the key icon "
            "(Secrets) in the left sidebar, add it, and switch on 'Notebook access'."
        )
    return value

HF_TOKEN = colab_secret("HF_TOKEN")
HF_DATASET_REPO = HF_DATASET_REPO or colab_secret("HF_DATASET_REPO")
print("Dataset repo:", HF_DATASET_REPO)

In [ ]:
%pip install -q "huggingface_hub>=0.23.0" pyarrow "chronos-forecasting>=2.3,<3"

In [ ]:
import io
import os
import subprocess
import sys
import tempfile
import time
from pathlib import Path

REPO_DIR = "/content/DEXTRA"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", GITHUB_REPO_URL, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
import torch
from huggingface_hub.utils import disable_progress_bars, enable_progress_bars
from src import config
from src.data import assemble
from src.evaluation import alarms, gap
from src.models import baselines, foundation, itransformer, met_moe, rare_days, sequences, training, windows
from src.utils.hf_store import HFStore

disable_progress_bars()          # dozens of downloads: no bar for each
device = itransformer.pick_device()
store = HFStore(HF_DATASET_REPO, token=HF_TOKEN)
daily = store.read_df("processed/city_daily_features.parquet")

# Recomputed from the current config rather than trusted as saved.
saved = daily["split"].copy() if "split" in daily.columns else None
daily["split"] = assemble.assign_split(daily)
differing = int((saved != daily["split"]).sum()) if saved is not None else 0
if differing:
    print(f"! {differing:,} rows differ from the saved splits; re-run notebook 02.")

# The 12-hour ceiling counts from when this VM booted, not from this cell.
try:
    session_age = float(open("/proc/uptime").read().split()[0])
except (OSError, ValueError, IndexError):
    session_age = 0.0
if not 0.0 <= session_age < 12 * 3600:   # not a Colab VM's age: count from now instead
    session_age = 0.0
deadline = time.monotonic() + max(0.0, TIME_BUDGET_HOURS * 3600 - session_age)

registry = training.RunRegistry(store, "runs/06")
cities = sorted(daily.city.unique()) if LOCO_CITIES is None else list(LOCO_CITIES)
fold_run = lambda city: "loco-" + city.replace(" ", "_")
planned = ([f"fm-{name}" for name, _, _ in FOUNDATION]
           + [f"main-{fix}_seed{seed}" for fix in FIXES for seed in SEEDS]
           + [fold_run(city) for city in cities])
finished = [name for name in planned if registry.done(name)]

print(f"device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))
if device.type != "cuda":
    print("! No GPU: this will take many hours rather than two. "
          "Runtime -> Change runtime type -> T4 GPU, then Run all.")
print(f"{len(daily):,} city-days, {daily.city.nunique()} cities")
print(f"{len(finished)} of {len(planned)} pieces already finished"
      + (" -- resuming where the last session stopped" if finished else ""))
print(f"time budget left this session: {(deadline - time.monotonic()) / 3600:.1f} h")

## A · Two protocols, one set of windows
The windows are notebook 05's: 14 days of every variate in, three days of AQI out. Two ways of holding cities out:

- **Main split**: ten training cities, five unseen ones, scored on 2025, with alarm levels tuned on the training cities' 2024.
- **Leave-one-city-out**: each of the fifteen cities held out in turn. A model trains on the other fourteen (2019–2023), tunes on their 2024, and is scored on the held-out city's 2024 *and* 2025.

Below, how many severe targets each protocol has to score.

In [ ]:
X, y, index = sequences.build_sequences(daily)
variates = sequences.feature_names(daily)
target_index = variates.index(sequences.TARGET)
gate_indices, gate_logged = met_moe.gate_layout(variates)
is_train, is_val = (index.split == "train").to_numpy(), (index.split == "val").to_numpy()
HORIZONS = [f"h{h + 1}" for h in range(y.shape[1])]

# Every forecast whose answers fall in 2024-2025: the main split's validation and test
# rows, and every city's leave-one-city-out test rows. Every forecaster scores them all.
scored = (index.last_target_date > pd.Timestamp(config.TRAIN_END_DATE)).to_numpy()
origins = index.loc[scored, ["city", "date"]].reset_index(drop=True)
today = X[:, -1, target_index]

pool = gap.long_frame(index[scored], y[scored], today[scored], {})
is_severe = pool.y > config.RARE_AQI_THRESHOLD
severe = pd.DataFrame({
    "role in the main split": pd.Series({c: "training city" if c in config.TRAIN_CITIES else "unseen city"
                                         for c in cities}),
    "severe targets 2024-25": pool[is_severe].groupby("city").size(),
    "peak AQI 2024-25": pool.groupby("city").y.max().round(),
}).reindex(cities)
severe["severe targets 2024-25"] = severe["severe targets 2024-25"].fillna(0).astype(int)
display(severe.sort_values("severe targets 2024-25", ascending=False))
print(f"main split, unseen cities, 2025: {int((is_severe & (pool.split == 'test_unseen')).sum())} severe targets")
print(f"leave-one-city-out, every city held out once, 2024-25: "
      f"{int(severe['severe targets 2024-25'].sum())} severe targets")

## B · Foundation models, zero-shot
Three setups, none trained on anything here:

- **Chronos-Bolt** (Amazon, base, 205M parameters): each city's AQI history alone.
- **Chronos-2** (Amazon, 120M): the same history.
- **Chronos-2 with weather**: the same, plus the 22 measured variables our models read (pollutants, weather, fire, calendar) as past covariates.

Each forecast reads up to 512 days of the city's own history, ending on the day it is issued — far more than the fortnight our models get, because long context is what these models were built for. They forecast quantiles: the median is their forecast, and the 90th percentile is offered as an alarm forecast, the role the quantile fix plays in section C.

In [ ]:
filled = sequences.fill_gaps(daily, variates, sequences.pooled_training_medians(daily, variates))
covariates = [v for v in variates if v != sequences.TARGET and v not in config.CITY_META_FEATURES]
LEVELS = (0.1, 0.5, 0.9)


def save_run(name, result, frames=None, weights=None):
    """Record a finished piece with its forecasts (and weights) in one commit."""
    if not SAVE_TO_HF:
        return
    with tempfile.TemporaryDirectory() as tmp:
        attachments = {}
        for filename, frame in (frames or {}).items():
            attachments[filename] = Path(tmp) / filename
            frame.to_parquet(attachments[filename], index=False)
        if weights is not None:
            attachments[f"{name}.pt"] = Path(tmp) / f"{name}.pt"
            torch.save(weights, attachments[f"{name}.pt"])
        registry.record(name, result, attachments=attachments)


def load_frame(name):
    frame = pd.read_parquet(io.BytesIO(store.read_bytes(registry.attachment(f"{name}.parquet"))))
    frame["date"] = pd.to_datetime(frame["date"]).astype("datetime64[ns]")
    return frame


stopped, loaded = False, {}
for name, model_id, with_weather in FOUNDATION:
    run = f"fm-{name}"
    if registry.done(run):
        print(f"\u2713 {name} already forecast")
        continue
    if time.monotonic() > deadline:
        print("\u23f8 time budget reached -- Runtime -> Disconnect and delete runtime, then Run all")
        stopped = True
        break
    print(f"\n\u25b6 {name}  ({model_id}{', with weather' if with_weather else ''})")
    started = time.monotonic()
    if model_id not in loaded:
        loaded.clear()
        if device.type == "cuda":
            torch.cuda.empty_cache()
        enable_progress_bars()           # the one download worth watching
        loaded[model_id] = foundation.load(model_id, device)
        disable_progress_bars()
    quantiles = foundation.forecast(loaded[model_id], filled, origins, sequences.TARGET,
                                    covariates if with_weather else (), length=FM_CONTEXT,
                                    horizon=len(HORIZONS), levels=LEVELS,
                                    batch_size=1024 if "chronos-2" in model_id else 256)
    frame = pd.concat([origins.assign(horizon=h + 1, q10=quantiles[:, h, 0], q50=quantiles[:, h, 1],
                                      q90=quantiles[:, h, 2]) for h in range(len(HORIZONS))],
                      ignore_index=True)
    minutes = round((time.monotonic() - started) / 60, 2)
    save_run(run, {"model_id": model_id, "weather": with_weather, "context": FM_CONTEXT,
                   "forecasts": len(frame), "minutes": minutes}, frames={f"{run}.parquet": frame})
    print(f"  \u2191 recorded -- {len(origins):,} forecasts x {len(HORIZONS)} horizons, {minutes} min")
loaded.clear()

## C · The simple fixes, on the main split
Notebook 05's all-token head is the starting point: `plain` is its three trained seeds, reused rather than retrained. Each fix trains the same head from the same seeds:

- **oversampled**: windows holding a severe day drawn until they are a quarter of every epoch — C-STDA's target tail share, made of real windows repeated rather than new ones generated.
- **weighted**: the data unchanged, severe targets weighted to carry a quarter of the loss.
- **quantile**: trained for the 90th percentile by pinball loss, and early-stopped on validation pinball loss rather than RMSE, since its job is to sit high.

A fourth fix needs no training: an alarm level tuned on validation. Section E applies it to every forecaster alike.

In [ ]:
def build_head():
    return met_moe.MetMoE(X.shape[2], config.LOOKBACK_WINDOW, config.FORECAST_HORIZON, target_index,
                          gate_indices, gate_logged, router="none", d_model=D_MODEL, n_heads=N_HEADS,
                          n_layers=N_LAYERS, dropout=DROPOUT)


def fit(fix, seed, X_train, y_train, val_loader, checkpoint=None, log=print):
    """One all-token head trained one way. The seed fixes the start, so fixes differ only in training."""
    itransformer.set_seed(seed)
    model = build_head()
    if fix == "weighted":
        model = rare_days.WeightedSquaredError(model, rare_days.severe_weight(y_train))
    elif fix == "quantile":
        model = rare_days.QuantileForecaster(model)
    loader = (rare_days.oversampling_loader(X_train, y_train, BATCH_SIZE) if fix == "oversampled"
              else itransformer.make_loader(X_train, y_train, BATCH_SIZE, shuffle=True))
    checkpointer = (training.Checkpointer(store, checkpoint, every_seconds=CHECKPOINT_EVERY_MINUTES * 60)
                    if checkpoint else None)
    return training.train(model, loader, val_loader, device, epochs=EPOCHS, lr=LEARNING_RATE,
                          weight_decay=WEIGHT_DECAY, patience=PATIENCE, checkpointer=checkpointer,
                          deadline=deadline, save_finished=False, log=log,
                          score=rare_days.pinball_score() if fix == "quantile" else None)


def best_epoch(history):
    metric = "val_score" if history and "val_score" in history[0] else "val_rmse"
    return min(history, key=lambda h: h[metric])["epoch"] if history else 0


def aligned(frame):
    """A saved run's forecasts, in the order of this session's scored windows."""
    return origins.merge(frame, on=["city", "date"], how="left")[HORIZONS].to_numpy()


val_loader = itransformer.make_loader(X[is_val], y[is_val], BATCH_SIZE * 2)
main = {}                       # forecaster column -> forecasts for the scored windows
jobs = [(fix, seed) for fix in FIXES for seed in SEEDS]
for seed in SEEDS:
    weights = f"runs/05/none_seed{seed}.pt"
    if store.exists(weights):
        head = build_head()
        head.load_state_dict(torch.load(io.BytesIO(store.read_bytes(weights)), map_location="cpu",
                                        weights_only=True), strict=True)
        main[f"plain_seed{seed}"] = itransformer.predict(head.to(device), X[scored], device)
        print(f"\u2713 plain_seed{seed}: notebook 05's all-token head")
    else:
        jobs.insert(0, ("plain", seed))          # not in notebook 05: train it here, the same way

for fix, seed in jobs:
    run, column = f"main-{fix}_seed{seed}", f"{fix}_seed{seed}"
    if registry.done(run):
        main[column] = aligned(load_frame(run))
        print(f"\u2713 {run} already finished")
        continue
    if stopped or time.monotonic() > deadline:
        print("\u23f8 time budget reached -- Runtime -> Disconnect and delete runtime, then Run all")
        stopped = True
        break
    print(f"\n\u25b6 {run}")
    started = time.monotonic()
    model, history, outcome = fit(fix, seed, X[is_train], y[is_train], val_loader,
                                  checkpoint=f"checkpoints/06/{run}.pt")
    if outcome == "out_of_time":
        print("\u23f8 saved partway -- Runtime -> Disconnect and delete runtime, then Run all to resume it")
        stopped = True
        break
    main[column] = itransformer.predict(model, X[scored], device)
    minutes = round((time.monotonic() - started) / 60, 2)
    save_run(run, {"fix": fix, "seed": seed, "outcome": outcome, "best_epoch": best_epoch(history),
                   "epochs_run": len(history), "minutes": minutes, "history": history},
             frames={f"{run}.parquet": origins.assign(**{h: main[column][:, i] for i, h in enumerate(HORIZONS)})},
             weights=getattr(model, "inner", model).state_dict())
    print(f"  \u2191 recorded -- {outcome}, best epoch {best_epoch(history)}, {minutes} min")

## D · Leave-one-city-out
Each fold holds out one city. The windows are rebuilt with that city outside the training split, so even the medians that fill gaps never see it. `plain` and every fix train from each seed on the other fourteen cities' 2019–2023, alarm levels come from their 2024, and the held-out city's 2024–2025 is scored. Gradient boosting is refitted per fold on the same rows. Twelve runs a fold, fifteen folds — about ninety minutes on a T4 — and each fold is saved whole when it ends.

In [ ]:
# Gradient boosting's samples do not depend on the fold, only their labels do: built once here.
samples = windows.usable(windows.make_supervised(daily))
features = windows.feature_matrix(samples)
samples = samples.drop(columns="split")

for city in cities:
    run = fold_run(city)
    if registry.done(run):
        print(f"\u2713 {run} already finished")
        continue
    if stopped or time.monotonic() > deadline:
        print("\u23f8 time budget reached -- Runtime -> Disconnect and delete runtime, then Run all")
        stopped = True
        break
    print(f"\n\u25b6 {run}")
    started = time.monotonic()
    labels = assemble.leave_one_out_split(daily, city)
    Xf, yf, indexf = sequences.build_sequences(daily.assign(split=labels))
    train_rows, val_rows, test_rows = ((indexf.split == s).to_numpy() for s in ("train", "val", "test_loco"))
    keep = val_rows | test_rows
    fold_val = itransformer.make_loader(Xf[val_rows], yf[val_rows], BATCH_SIZE * 2)
    forecasts, runs, outcome = {}, [], None
    for fix in ["plain"] + FIXES:
        epochs = []
        for seed in SEEDS:
            model, history, outcome = fit(fix, seed, Xf[train_rows], yf[train_rows], fold_val,
                                          log=lambda *a: None)
            if outcome == "out_of_time":
                break
            forecasts[f"{fix}_seed{seed}"] = itransformer.predict(model, Xf[keep], device)
            runs.append({"fix": fix, "seed": seed, "outcome": outcome, "best_epoch": best_epoch(history),
                         "epochs_run": len(history)})
            epochs.append(str(best_epoch(history)))
        if outcome == "out_of_time":
            break
        print(f"  {fix:12s} best epochs {', '.join(epochs)}")
    if outcome == "out_of_time":
        print("\u23f8 time budget reached mid-fold -- this fold starts again next session")
        stopped = True
        break
    frame = gap.long_frame(indexf[keep], yf[keep], Xf[keep, -1, target_index], forecasts)
    by_target = daily[["city", "date"]].assign(split=labels).rename(columns={"date": "target_date"})
    fold_samples = samples.merge(by_target, on=["city", "target_date"], how="left")
    booster = baselines.fit_gradient_boosting(fold_samples[fold_samples.split == "train"], features)
    rows = fold_samples[fold_samples.split.isin(["val", "test_loco"])].copy()
    rows["gbm"] = baselines.predict_gradient_boosting(booster, rows, features).to_numpy()
    frame = frame.merge(rows[["city", "date", "horizon", "gbm"]], on=["city", "date", "horizon"], how="left")
    minutes = round((time.monotonic() - started) / 60, 2)
    save_run(run, {"city": city, "runs": runs, "windows": int(keep.sum()), "minutes": minutes},
             frames={f"{run}.parquet": frame})
    print(f"  \u2191 recorded -- {len(runs)} runs and gradient boosting, {minutes} min")

## E · The gap, measured
Every forecaster is scored the same way, on the rows every one of them has a forecast for: the alarm level is chosen on validation and read on the test rows — in leave-one-city-out, fold by fold. `persistence at 300` is persistence with the official line and no tuning at all. Counts are given beside the rates, and the bias on severe days in AQI points. For a foundation model, `median` is its forecast and `q90` its alarm forecast; the quantile fix's own forecast is already its 90th percentile. Those high-side forecasts sit above the truth by design, so their RMSE is left blank rather than shown as if it meant the same thing.

The last table puts the gap in one place: for each of the ten main-split training cities, the same city and the same year (2025), scored while the city is in training (main split) and while it is held out (its fold). The held-out model trains on fourteen cities rather than ten, so if it still does worse, the cost of being unseen is at least that large.

In [ ]:
def with_foundation(frame):
    """Every finished foundation model's median and 90th percentile, as forecaster columns."""
    for name, _, _ in FOUNDATION:
        if registry.done(f"fm-{name}"):
            fm = load_frame(f"fm-{name}").rename(columns={"q50": f"{name} median", "q90": f"{name} q90"})
            frame = frame.merge(fm[["city", "date", "horizon", f"{name} median", f"{name} q90"]],
                                on=["city", "date", "horizon"], how="left")
    return frame


gbm_main = store.read_df("processed/baseline_forecasts.parquet")[["city", "date", "horizon", "gbm"]]
gbm_main["date"] = pd.to_datetime(gbm_main["date"]).astype("datetime64[ns]")
main_frame = with_foundation(gap.long_frame(index[scored], y[scored], today[scored], main)
                             .merge(gbm_main, on=["city", "date", "horizon"], how="left"))
folds = {city: with_foundation(load_frame(fold_run(city))) for city in cities if registry.done(fold_run(city))}

NOT_FORECASTS = {"city", "date", "split", "horizon", "y"}
columns = [c for c in main_frame.columns if c not in NOT_FORECASTS]
loco_columns = [c for c in columns if folds and all(c in f.columns for f in folds.values())]
main_frame = main_frame.dropna(subset=columns)
folds = {city: f.dropna(subset=loco_columns) for city, f in folds.items()}

ORDER = (["persistence at 300", "persistence", "gbm", "plain"] + FIXES
         + [f"{name} {kind}" for name, _, _ in FOUNDATION for kind in ("median", "q90")])
HIGH_SIDE = {"quantile"} | {f"{name} q90" for name, _, _ in FOUNDATION}
SHOW = ["rmse", "hits", "severe", "false alarms", "f1 at 300", "f1", "f1 sd", "bias on severe days", "alarm level"]
LINE = config.RARE_AQI_THRESHOLD


def report(results, title):
    summary, counts = gap.combine_seeds(results)
    summary = summary.reindex([name for name in ORDER if name in summary.index])
    summary.loc[summary.index.isin(HIGH_SIDE), "rmse"] = np.nan     # meant to sit high: not a fit
    print(f"\n{title}")
    display(summary[SHOW].round(3))
    return summary, counts


tune = main_frame.split == "val"


def main_results(test):
    results = {c: gap.score_column(main_frame, c, tune, test) for c in columns}
    results["persistence at 300"] = gap.score_column(main_frame, "persistence", tune, test, level=LINE)
    return results


seen, _ = report(main_results(main_frame.split == "test_seen"), "Main split, the ten training cities, 2025")
unseen, _ = report(main_results(main_frame.split == "test_unseen"), "Main split, the five unseen cities, 2025")

loco, loco_counts = None, {}
if folds:
    loco_results = {c: gap.score_folds(folds, c) for c in loco_columns}
    loco_results["persistence at 300"] = gap.score_folds(folds, "persistence", level=LINE)
    loco, loco_counts = report(loco_results, f"Leave-one-city-out, {len(folds)} held-out cities, 2024-25")

if folds:
    intervals = {name: alarms.bootstrap_groups(counts) for name, counts in loco_counts.items()}
    print("95% intervals for the pooled F1, resampling whole held-out cities:")
    for name in [n for n in ORDER if n in intervals]:
        low, high = intervals[name]
        print(f"  {name:24s} {loco.loc[name, 'f1']:.3f}  [{low:.3f}, {high:.3f}]")

    per_city = pd.DataFrame({name: (2 * c.hits / (c.severe + c.raised)).set_axis(c.city).round(3)
                             for name, c in loco_counts.items()})
    per_city.insert(0, "severe targets", loco_counts["persistence"].set_index("city").severe.astype(int))
    print("\nSevere-day F1 for each city when it is the one held out:")
    display(per_city[["severe targets"] + [n for n in ORDER if n in per_city.columns]]
            .sort_values("severe targets", ascending=False))

    tested = pd.concat([f[f.split == "test_loco"] for f in folds.values()], ignore_index=True)
    for name in {gap.forecaster(c)[0] for c in loco_columns if gap.forecaster(c)[1] is not None}:
        tested[name] = tested[[c for c in loco_columns if gap.forecaster(c)[0] == name]].mean(axis=1)
    shown = [n for n in ORDER if n in tested.columns and n not in HIGH_SIDE]
    print("\nForecasts issued on a severe day in a held-out city: the mean truth, and the mean forecast")
    display(alarms.return_to_normal(tested, shown).round(1))

    # The same city, the same year: scored while in training, and while held out.
    target_year = lambda f: (f.date + pd.to_timedelta(f.horizon, unit="D")).dt.year
    held_in = [c for c in folds if c in config.TRAIN_CITIES]
    rows = []
    for column in [c for c in loco_columns if gap.forecaster(c)[0] in ("persistence", "plain")]:
        for city in held_in:
            fold = folds[city]
            inside, _ = gap.score_column(main_frame, column, tune, (main_frame.split == "test_seen")
                                         & (main_frame.city == city) & (target_year(main_frame) == 2025))
            outside, _ = gap.score_column(fold, column, fold.split == "val", (fold.split == "test_loco")
                                          & (target_year(fold) == 2025))
            rows.append({"forecaster": gap.forecaster(column)[0], "city": city,
                         "severe": inside["severe"], "hits in training": inside["hits"],
                         "raised in training": inside["hits"] + inside["false alarms"],
                         "hits held out": outside["hits"], "raised held out": outside["hits"] + outside["false alarms"],
                         "bias in training": inside["bias on severe days"],
                         "bias held out": outside["bias on severe days"]})
    if rows:
        cost = pd.DataFrame(rows).groupby(["forecaster", "city"]).mean(numeric_only=True)   # seeds averaged
        pooled = cost.groupby("forecaster").sum()
        f1 = lambda c, side: 2 * c[f"hits {side}"] / (c["severe"] + c[f"raised {side}"])
        weighted = lambda side: (cost[f"bias {side}"] * cost["severe"]).groupby("forecaster").sum() / pooled["severe"]
        summary = pd.DataFrame({"severe targets": pooled["severe"].astype(int),
                                "F1 in training": f1(pooled, "in training"), "F1 held out": f1(pooled, "held out"),
                                "bias in training": weighted("in training"), "bias held out": weighted("held out")})
        print(f"\nThe cost of being unseen: {len(held_in)} main-split training cities, 2025, "
              "in training vs held out")
        display(summary.round(3))
        per_city = cost.loc["plain"] if "plain" in cost.index.get_level_values(0) else None
        if per_city is not None:
            display(pd.DataFrame({"severe targets": per_city["severe"].astype(int),
                                  "F1 in training": f1(per_city, "in training"),
                                  "F1 held out": f1(per_city, "held out")}).round(3)
                    .sort_values("severe targets", ascending=False))

## F · Does anything close the gap?
The measure fixed at the top of the notebook: severe-day F1 on held-out cities, pooled over the leave-one-city-out folds, alarm tuned on validation. The bar is persistence at the better of its two alarm rules. For each forecaster, its difference from that bar and from the plain all-token head, with 95% intervals from resampling whole cities. A forecaster **beats persistence** only if its interval lies wholly above zero — and the size of the difference is the number to read: with few cities, a consistent sliver can clear zero without mattering.

In [ ]:
if not folds:
    print("Leave-one-city-out has not finished yet -- the verdict needs it. Run all again to continue.")
else:
    bar = max(["persistence at 300", "persistence"], key=lambda n: loco.loc[n, "f1"])
    print(f"the bar: {bar}, pooled F1 {loco.loc[bar, 'f1']:.3f} -- the better of persistence's two alarm rules")
    rows, over_plain = [], []
    for name in [n for n in ORDER if n in loco_counts and not n.startswith("persistence")]:
        versus = gap.paired_difference(loco_counts[name], loco_counts[bar])
        plain = (gap.paired_difference(loco_counts[name], loco_counts["plain"])
                 if "plain" in loco_counts and name != "plain" else (np.nan, np.nan, np.nan))
        if np.isfinite(plain[1]) and plain[1] > 0:
            over_plain.append(name)
        rows.append({"forecaster": name, "pooled F1": loco.loc[name, "f1"],
                     "vs persistence": versus[0], "95% interval": f"[{versus[1]:+.3f}, {versus[2]:+.3f}]",
                     "vs plain": plain[0],
                     "95% interval ": "" if np.isnan(plain[0]) else f"[{plain[1]:+.3f}, {plain[2]:+.3f}]",
                     "beats persistence": "yes" if versus[1] > 0 else "no"})
    verdict = pd.DataFrame(rows).set_index("forecaster")
    display(verdict.round(3))
    winners = verdict.index[verdict["beats persistence"] == "yes"].tolist()
    if winners:
        best = max(winners, key=lambda n: verdict.loc[n, "vs persistence"])
        print(f"\nBeat persistence on held-out cities (interval above zero): {', '.join(winners)}.")
        print(f"Largest margin: {best}, {verdict.loc[best, 'vs persistence']:+.3f} F1.")
    else:
        print("\nNothing beats persistence on held-out cities: neither a foundation model nor a simple fix "
              "detects their severe days better. That is the gap C-STDA has to close.")
    print("Better than the plain all-token head (interval above zero): "
          + (", ".join(over_plain) + " -- C-STDA has to beat these too." if over_plain
             else "none -- no fix or foundation model improves on it, so the plain head is C-STDA's bar."))

## G · Save
The scored forecasts of both protocols and every score table, for the paper's tables and for notebook 07, where C-STDA is measured against the best of what is here.

In [ ]:
if SAVE_TO_HF:
    tables = [t.assign(protocol=p) for t, p in ((seen, "main, training cities"), (unseen, "main, unseen cities"),
                                                (loco, "leave-one-city-out")) if t is not None]
    store.upload_df(pd.concat(tables).reset_index(), "processed/gap_scores.csv",
                    "Notebook 06: severe-day scores for every forecaster and protocol")
    store.upload_df(main_frame, "processed/gap_main_forecasts.parquet",
                    "Notebook 06: main-split forecasts from every forecaster")
    if folds:
        store.upload_df(pd.concat([f.assign(fold=c) for c, f in folds.items()], ignore_index=True),
                        "processed/gap_loco_forecasts.parquet",
                        "Notebook 06: leave-one-city-out forecasts from every forecaster")
    print("\u2191 processed/gap_scores.csv, processed/gap_main_forecasts.parquet"
          + (", processed/gap_loco_forecasts.parquet" if folds else ""))
print("\nNext: notebook 07 builds C-STDA and measures it against the bar set here.")